# Qwen3-VL caption-only structured events (prompt r3) — full 900-sample run

This notebook runs the **caption-only** structured event decomposition
(`preprocessing/qwen3vl_event_chunks_text.py`) for all 900 systematically
sampled captions. No frames are fed to the model: boundaries come from the
caption's linguistic cues with few-shot examples, and the array keeps the
caption's narration order. The output schema
(`Id, Sentence, Events, Camera, Progression`) is identical to the
image-conditioned runs, so surveys stay directly comparable.

The bundle is small (~2 MiB — code plus `train.csv`, no images). Resumable
state and the final CSV are written directly to Google Drive. Text-only
generation is much faster than the image runs.

After completion, copy the Drive output folder back into the local repo at
`preprocessing/generated/qwen3vl_event_chunks_v5_text_900/`.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

DRIVE_ROOT = Path('/content/drive/MyDrive/snu ai')
BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_qwen3vl_event_chunks_v5_text_bundle.tar'
EXPECTED_BUNDLE_SHA256 = '8F2A8D61DB8CA99A42786CEFEF13E8B7A956D3D9F92A60F1A83E38C69330856F'
WORKDIR = Path('/content/snu_ai')
TRAIN_CSV = WORKDIR / 'snuaichallenge_data/train.csv'
OUTPUT_DIR = DRIVE_ROOT / 'preprocessing/qwen3vl_event_chunks_v5_text_900'
HF_HOME = Path('/content/huggingface')
os.environ['HF_HOME'] = str(HF_HOME)

SAMPLE_COUNT = 900
SEED = 42
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_REVISION = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'
QUANTIZATION = 'auto'
BATCH_SIZE = 0
MAX_NEW_TOKENS = 384
MAX_RETRIES = 1


## 1. Mount Drive, install dependencies, and extract the bundle

Upload `snu_colab_qwen3vl_event_chunks_v5_text_bundle.tar` to
`MyDrive/snu ai/` before running this cell.


In [2]:
import hashlib
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46',
], check=True)

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(
        f'Upload the bundle to this exact path: {BUNDLE_PATH}'
    )
bundle_sha256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest().upper()
if bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        f'Bundle checksum mismatch: {bundle_sha256} != {EXPECTED_BUNDLE_SHA256}'
    )

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xf', str(BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
if not TRAIN_CSV.is_file():
    raise FileNotFoundError(TRAIN_CSV)
sys.path.insert(0, str(WORKDIR))
print('Bundle verified and extracted:', bundle_sha256)
print('train.csv:', TRAIN_CSV, TRAIN_CSV.stat().st_size, 'bytes')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bundle verified and extracted: 8F2A8D61DB8CA99A42786CEFEF13E8B7A956D3D9F92A60F1A83E38C69330856F
train.csv: /content/snu_ai/snuaichallenge_data/train.csv 2136130 bytes


## 2. CPU preflight: prompt formatting and chat template

Downloads only the small processor/tokenizer files, not the 17.5 GB model.


In [3]:
from transformers import AutoProcessor

from preprocessing.qwen3vl_event_chunks_text import (
    build_messages,
    _prompt_sha256,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
conversation = build_messages('A preflight caption.')
inputs = processor.apply_chat_template(
    [conversation],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors='pt',
    processor_kwargs={'padding': True},
)
assert 'input_ids' in inputs
assert 'pixel_values' not in inputs
print('input_ids shape:', tuple(inputs['input_ids'].shape))
print('prompt_sha256:', _prompt_sha256())
print('Preflight passed: caption-only messages tokenize cleanly.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

input_ids shape: (1, 886)
prompt_sha256: 90168904e13b234f77f5eeedccdfa2dfc8c20b5677b4d4b0aacb81c426f08028
Preflight passed: caption-only messages tokenize cleanly.


## 3. Prepare and verify the 900-caption sample

CPU-only. Re-running is safe and preserves compatible Drive state.


In [4]:
import csv
import json

prepare_command = [
    sys.executable, '-m', 'preprocessing.qwen3vl_event_chunks_text',
    '--train-csv', str(TRAIN_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--sample-count', str(SAMPLE_COUNT),
    '--seed', str(SEED),
    '--model-id', MODEL_ID,
    '--revision', MODEL_REVISION,
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-retries', str(MAX_RETRIES),
    '--prepare-only',
]
subprocess.run(prepare_command, check=True, cwd=WORKDIR)

sample_path = OUTPUT_DIR / 'sampled_captions.csv'
metadata_path = OUTPUT_DIR / 'metadata.json'
with sample_path.open('r', encoding='utf-8', newline='') as file:
    sampled_rows = list(csv.DictReader(file))
assert len(sampled_rows) == SAMPLE_COUNT, len(sampled_rows)
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
assert metadata['validator_version'] == 3, metadata['validator_version']
assert metadata['sample_count'] == SAMPLE_COUNT
assert metadata['model']['caption_only'] is True
print('prepared', len(sampled_rows), 'captions for the caption-only run')


prepared 900 captions for the caption-only run


## 4. Inspect the inference GPU and selected loading mode

Inference-only. Full BF16/FP16 weights need at least 22 GiB VRAM; below that
the script selects NF4 4-bit loading. Text-only batches are larger than the
image runs (8 at 40 GiB and above).


In [5]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
from preprocessing.qwen3vl_event_chunks_text import _gpu_configuration

gpu_config = _gpu_configuration(QUANTIZATION, BATCH_SIZE)
print('GPU:', gpu_config['name'])
print(f"VRAM: {gpu_config['total_vram_gib']:.2f} GiB")
print('compute capability:', gpu_config['compute_capability'])
print('dtype:', gpu_config['dtype_name'])
print('quantization:', gpu_config['quantization'])
print('generation batch size:', gpu_config['batch_size'])


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 94.97 GiB
compute capability: 12.0
dtype: bfloat16
quantization: none
generation batch size: 8


## 5. Run or resume the caption-only generation

The script restores compatible rows from `state.jsonl` first, then generates
only pending Ids. Every Id is recorded: after one correction retry a failed
sample falls back to a single caption-copy event and is logged in
`errors.jsonl`.


In [6]:
from collections import deque
from datetime import datetime, timezone

run_command = [
    sys.executable, '-m', 'preprocessing.qwen3vl_event_chunks_text',
    '--train-csv', str(TRAIN_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--sample-count', str(SAMPLE_COUNT),
    '--seed', str(SEED),
    '--model-id', MODEL_ID,
    '--revision', MODEL_REVISION,
    '--quantization', QUANTIZATION,
    '--batch-size', str(BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-retries', str(MAX_RETRIES),
]

state_path = OUTPUT_DIR / 'state.jsonl'
errors_path = OUTPUT_DIR / 'errors.jsonl'
fatal_error_path = OUTPUT_DIR / 'fatal_error.log'
log_path = OUTPUT_DIR / 'qwen_run.log'

def nonempty_line_count(path):
    if not path.is_file():
        return 0
    with path.open('r', encoding='utf-8') as file:
        return sum(bool(line.strip()) for line in file)

print('resuming valid state rows:', nonempty_line_count(state_path))
print('previous error records:', nonempty_line_count(errors_path))
print('full streaming log:', log_path)
print('fatal traceback log:', fatal_error_path)

tail = deque(maxlen=120)
with log_path.open('a', encoding='utf-8', buffering=1) as log_file:
    marker = f"\n===== run {datetime.now(timezone.utc).isoformat()} =====\n"
    log_file.write(marker)
    print(marker, end='')
    process = subprocess.Popen(
        run_command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors='replace',
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        tail.append(line)
    return_code = process.wait()

if return_code != 0:
    print('\n===== INNER ERROR TAIL =====')
    print(''.join(tail), end='')
    raise RuntimeError(
        f'Caption-only event generation failed with exit code {return_code}. '
        f'The real traceback is printed above and saved at {log_path} '
        f'and {fatal_error_path}'
    )
print('Caption-only 900-sample event generation completed successfully.')


resuming valid state rows: 0
previous error records: 0
full streaming log: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_v5_text_900/qwen_run.log
fatal traceback log: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_v5_text_900/fatal_error.log

===== run 2026-07-20T11:14:30.834942+00:00 =====
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB), dtype=bfloat16, quantization=none, batch_size=8

Fetching 4 files: 100%|██████████| 4/4 [00:47<00:00, 11.77s/it]

Loading weights: 100%|██████████| 750/750 [00:01<00:00, 595.49it/s]
completed=8/900 run_processed=8/900 rate=1.38 captions/s eta=10.8 min
completed=16/900 run_processed=16/900 rate=1.61 captions/s eta=9.2 min
completed=24/900 run_processed=24/900 rate=1.80 captions/s eta=8.1 min
completed=32/900 run_processed=32/900 rate=1.97 captions/s eta=7.3 min
completed=40/900 run_processed=40/900 rate=2.09 captions/s eta=6.8 min
completed=48/900 run_processed=48/900 rate=2.14 captions/s eta=6.6 min
co

## 6. Validate the final file and inspect the structured events


In [7]:
from collections import Counter

import pandas as pd

events_path = OUTPUT_DIR / 'events.csv'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
assert metadata['status'] == 'complete', metadata
with events_path.open('r', encoding='utf-8', newline='') as file:
    reader = csv.DictReader(file)
    event_rows = list(reader)
    fieldnames = reader.fieldnames

assert fieldnames == ['Id', 'Sentence', 'Events', 'Camera', 'Progression'], fieldnames
assert len(event_rows) == SAMPLE_COUNT
assert [row['Id'] for row in event_rows] == [row['Id'] for row in sampled_rows]

state_records = {}
with (OUTPUT_DIR / 'state.jsonl').open('r', encoding='utf-8') as file:
    for line in file:
        if line.strip():
            payload = json.loads(line)
            state_records[payload['Id']] = payload
fallback_ids = sorted(
    sample_id
    for sample_id, payload in state_records.items()
    if payload.get('fallback')
)
retry_ids = sorted(
    sample_id
    for sample_id, payload in state_records.items()
    if payload.get('attempts', 1) > 1 and not payload.get('fallback')
)

decoded = []
camera_events = 0
progression_events = 0
total_events = 0
for row in event_rows:
    events = json.loads(row['Events'])
    cameras = json.loads(row['Camera'])
    progressions = json.loads(row['Progression'])
    assert 1 <= len(events) <= 4
    assert len(events) == len(cameras) == len(progressions)
    total_events += len(events)
    camera_events += sum(camera != 'none' for camera in cameras)
    progression_events += sum(p is not None for p in progressions)
    decoded.append({**row, 'DecodedEvents': events, 'EventCount': len(events)})

print('validated final rows:', len(decoded))
print('event-count distribution:', dict(sorted(Counter(
    row['EventCount'] for row in decoded
).items())))
print(f'camera != none: {camera_events}/{total_events}')
print(f'progression != null: {progression_events}/{total_events}')
print('fallback ids:', fallback_ids)
print('retry ids (attempts > 1):', retry_ids)
print('final CSV:', events_path)
review = pd.DataFrame([
    {
        'Id': row['Id'],
        'Sentence': row['Sentence'],
        'Events': '\n'.join(
            f'{index + 1}. [{camera}] {event}'
            + (f' | progression: {progression}' if progression else '')
            for index, (event, camera, progression) in enumerate(zip(
                row['DecodedEvents'],
                json.loads(row['Camera']),
                json.loads(row['Progression']),
            ))
        ),
    }
    for row in decoded
]).sample(n=min(20, len(decoded)), random_state=SEED)
display(review)


validated final rows: 900
event-count distribution: {1: 167, 2: 439, 3: 200, 4: 94}
camera != none: 311/2021
progression != null: 185/2021
fallback ids: ['JqutMU', 'iaRnOP', 'wQMMKq']
retry ids (attempts > 1): ['2qOqKv', '7dQ6qc', 'G2J9QY', 'a3I7Jc']
final CSV: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_v5_text_900/events.csv


,Id,Sentence,Events
70,BmqcGY,"A hand adjusts a control panel, then a woman w...",1. [none] A hand adjusts a control panel.\n2. ...
827,iN9SBo,A little girl is guided as she mounts a camel ...,1. [none] A little girl is guided as she mount...
231,BdoK2L,"The person adjusts a yellow sock on their leg,...",1. [none] A person adjusts a yellow sock on th...
588,sGuQI8,The people then begin shaving the other person...,1. [none] The people are shaving the other per...
39,gWyXHL,The scene transitions from a peeled potato and...,1. [none] A peeled potato and peeler are shown...
731,6uWrfu,The camera zooms out slightly as the guitarist...,1. [zoom_out] The guitarist's hand shifts down...
299,twjKu6,"The girl begins to twirl a baton, then moves i...",1. [none] The girl begins to twirl a baton.\n2...
110,MJNBT4,The girl in a pink hoodie stands at a sink as ...,1. [none] A girl in a pink hoodie stands at a ...
72,XGrfcQ,The person crouches to inspect the bike's comp...,1. [none] The person crouches to inspect the b...
86,F84SDO,The person tightens the handlebars and lowers ...,1. [none] The person tightens the handlebars a...


## Output

The run writes to
`/content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_v5_text_900/`:
`events.csv`, `state.jsonl`, `errors.jsonl`, `metadata.json`, `qwen_run.log`.

Copy that whole folder back into the local repo as
`preprocessing/generated/qwen3vl_event_chunks_v5_text_900/`. Its `events.csv`
(900 rows, same five-column schema) feeds the next GPT survey and the CLIP
scoring comparison against the image-conditioned runs.
